<a href="https://www.kaggle.com/code/ahmad03038/cristiano-ronaldo-youtube-stats-daily-pull?scriptVersionId=354535451" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

## Import necessary libraries

In [1]:
import glob
import os
import re
from datetime import datetime, timezone

import pandas as pd
from googleapiclient.discovery import build
from kaggle_secrets import UserSecretsClient  # Securely access API key

<h2 style="color: #8B0000;">  Find The Youtube Channel ID of Cristiano Ronaldo Using Youtube API: </h2>


In [2]:
CHANNEL_ID = "UCtxD0x6AuNNqdXO9Wp5GHew"      # @cristiano
FILENAME = "cristiano_youtube_stats.csv"
INPUT_ROOT = "/kaggle/input"                 # where attached datasets are mounted
WORKING_DIR = "/kaggle/working"              # notebook output -> becomes the dataset file

ALLOW_FRESH_START = False   # set True ONLY for the very first run of a brand-new dataset
MIN_COVERAGE = 0.90         # abort if today's pull has < 90% of the videos seen in the previous pull

# Fixed schema: keeps today's rows aligned with all historical rows
EXPECTED_COLUMNS = [
    "id", "title", "description", "publishTime", "kind_stats", "duration_seconds", "video_type",
    "viewCount", "likeCount", "commentCount",
    "thumbnails.default.url", "thumbnails.medium.url", "thumbnails.high.url",
    "contentDetails.duration", "contentDetails.dimension", "topicDetails.topicCategories",
    "snippet.defaultLanguage", "localizations.en.title", "localizations.en.description",
    "snippet.tags", "pull_date",
]

# Daily Updates (Pull of Data)

<h2 style="color: #8B0000;"> Step 1: Authenticate & Connect to YouTube API</h2>


In [3]:
def get_youtube_api():
    """
    Retrieves the YouTube API key securely from Kaggle Secrets and initializes the API client.
    """
    user_secrets = UserSecretsClient()
    API_KEY = user_secrets.get_secret("YOUTUBE_API_KEY_AHMED")  # Secure API Key
    return build("youtube", "v3", developerKey=API_KEY)

<h2 style="color: #8B0000;">   Step 2: Fetch Video IDs from Cristiano Ronaldo's YouTube Channel</h2>


In [4]:
def get_channel_videos(youtube, channel_id):
    """
    Returns the IDs of ALL videos on the channel (no cap), newest first.
    """
    uploads_playlist_id = "UU" + channel_id[2:]
    video_ids, page_token = [], None

    while True:
        response = youtube.playlistItems().list(
            part="contentDetails",
            playlistId=uploads_playlist_id,
            maxResults=50,
            pageToken=page_token,
        ).execute()

        video_ids.extend(item["contentDetails"]["videoId"] for item in response.get("items", []))

        page_token = response.get("nextPageToken")
        if not page_token:
            break

    return list(dict.fromkeys(video_ids))   # de-duplicate, keep order

<h2 style="color: #8B0000;">  Step 3: Convert YouTube Video Duration to Seconds</h2>


In [5]:
_DURATION_RE = re.compile(r"^P(?:(\d+)D)?(?:T(?:(\d+)H)?(?:(\d+)M)?(?:(\d+)S)?)?$")

def convert_duration(duration):
    """
    Converts YouTube video duration format (ISO 8601) to total seconds.
    e.g. 'PT5M20S' -> 320, 'PT6H8M45S' -> 22125, 'P1DT2H' -> 93600, 'P0D' -> 0
    """
    match = _DURATION_RE.match(duration or "")
    if not match:
        return 0
    days, hours, minutes, seconds = (int(g) if g else 0 for g in match.groups())
    return days * 86400 + hours * 3600 + minutes * 60 + seconds

<h2 style="color: #8B0000;">  Step 4: Fetch Video Details (Views, Likes, Comments, etc.)</h2>


In [6]:
def _to_int(value):
    """Counts as integers; None if YouTube hides them (e.g. likes or comments disabled)."""
    try:
        return int(value)
    except (TypeError, ValueError):
        return None


def get_video_details(youtube, video_ids, pull_date):
    """
    Retrieves detailed statistics and metadata for a list of video IDs.

    Parameters:
        youtube: YouTube API client
        video_ids (list): Video IDs
        pull_date (str): ONE date for the whole run, e.g. "2026-10-02"
    """
    video_stats = []
    for i in range(0, len(video_ids), 50):  # Fetch stats in batches of 50
        batch = video_ids[i:i + 50]
        response = youtube.videos().list(
            part="snippet,statistics,contentDetails,topicDetails,localizations",
            id=",".join(batch),
        ).execute()

        returned = {item["id"] for item in response.get("items", [])}
        missing = [v for v in batch if v not in returned]
        if missing:
            print(f"⚠️ {len(missing)} video(s) not returned by the API (deleted/private?): {missing}")

        for item in response.get("items", []):
            snippet = item.get("snippet", {})
            statistics = item.get("statistics", {})
            content_details = item.get("contentDetails", {})
            topic_details = item.get("topicDetails", {})
            localizations = item.get("localizations", {}).get("en", {})

            duration_seconds = convert_duration(content_details.get("duration", ""))
            if duration_seconds == 0:
                video_type = "live"                      # live stream in progress / upcoming
            else:
                video_type = "long" if duration_seconds >= 60 else "short"   # note: Shorts can now be up to 3 min

            video_stats.append({
                "id": item["id"],
                "title": snippet.get("title", "N/A"),
                "description": snippet.get("description", "N/A"),
                "publishTime": snippet.get("publishedAt", "N/A"),
                "kind_stats": item.get("kind", "N/A"),
                "duration_seconds": duration_seconds,
                "video_type": video_type,
                "viewCount": _to_int(statistics.get("viewCount")),
                "likeCount": _to_int(statistics.get("likeCount")),
                "commentCount": _to_int(statistics.get("commentCount")),
                "thumbnails.default.url": snippet.get("thumbnails", {}).get("default", {}).get("url", "N/A"),
                "thumbnails.medium.url": snippet.get("thumbnails", {}).get("medium", {}).get("url", "N/A"),
                "thumbnails.high.url": snippet.get("thumbnails", {}).get("high", {}).get("url", "N/A"),
                "contentDetails.duration": content_details.get("duration", "N/A"),
                "contentDetails.dimension": content_details.get("dimension", "N/A"),
                "topicDetails.topicCategories": topic_details.get("topicCategories", "N/A"),
                "snippet.defaultLanguage": snippet.get("defaultLanguage", "N/A"),
                "localizations.en.title": localizations.get("title", "N/A"),
                "localizations.en.description": localizations.get("description", "N/A"),
                "snippet.tags": snippet.get("tags", "N/A"),
                "pull_date": pull_date,
            })
    return video_stats

<h2 style="color: #8B0000;"> Step 5: Save Data (Appending Latest Stats to Existing Dataset)</h2>


In [7]:
def find_existing_file(filename=FILENAME, root=None):
    """Finds the previous dataset file wherever Kaggle mounted it."""
    root = root or INPUT_ROOT
    hits = glob.glob(os.path.join(root, "**", filename), recursive=True)
    return max(hits, key=os.path.getsize) if hits else None


def load_existing(filename=FILENAME):
    path = find_existing_file(filename)
    if path:
        df = pd.read_csv(path, low_memory=False)
        df["pull_date"] = df["pull_date"].astype(str).str[:10]      # normalise any old timestamp rows
        missing = [c for c in EXPECTED_COLUMNS if c not in df.columns]
        extra = [c for c in df.columns if c not in EXPECTED_COLUMNS]
        if missing or extra:
            raise ValueError(f"Schema mismatch with {path}. Missing: {missing}. Unexpected: {extra}")
        print(f"📂 Loaded {len(df):,} existing rows ({df['pull_date'].nunique()} pull dates) from {path}")
        return df
    if ALLOW_FRESH_START:
        print("⚠️ No existing file found. Starting fresh because ALLOW_FRESH_START=True.")
        return pd.DataFrame(columns=EXPECTED_COLUMNS)
    raise FileNotFoundError(
        f"Previous '{filename}' not found under {INPUT_ROOT}. Add your dataset as an Input to this notebook. "
        "Refusing to continue: saving now would overwrite the history with a single day."
    )


def check_coverage(df_existing, df_new):
    """Stops a partial pull (API hiccup) from being saved as if it were a full snapshot."""
    if df_existing.empty:
        return
    last_date = df_existing["pull_date"].max()
    previous_n = df_existing.loc[df_existing["pull_date"] == last_date, "id"].nunique()
    new_n = df_new["id"].nunique()
    if new_n < MIN_COVERAGE * previous_n:
        raise RuntimeError(
            f"Today's pull has {new_n} videos but the previous pull ({last_date}) had {previous_n}. "
            "Looks incomplete, so nothing was saved."
        )


def merge_snapshots(df_existing, df_new):
    df_new = df_new.reindex(columns=EXPECTED_COLUMNS)
    combined = df_new if df_existing.empty else pd.concat([df_existing, df_new], ignore_index=True)
    combined["pull_date"] = combined["pull_date"].astype(str).str[:10]
    combined = combined.drop_duplicates(subset=["id", "pull_date"], keep="last")   # same-day rerun replaces
    return combined[EXPECTED_COLUMNS].sort_values(["pull_date", "id"], kind="stable").reset_index(drop=True)


def save_stats(stats, filename=FILENAME):
    """
    Appends today's snapshot to the existing history and writes the result to the working directory.
    """
    df_existing = load_existing(filename)
    df_new = pd.DataFrame(stats)
    check_coverage(df_existing, df_new)

    df_combined = merge_snapshots(df_existing, df_new)
    assert len(df_combined) >= len(df_existing), "Refusing to save: output would be smaller than the existing data."

    save_path = os.path.join(WORKING_DIR, filename)
    df_combined.to_csv(save_path, index=False)

    # Read back to be sure the file on disk is what we think it is
    check = pd.read_csv(save_path, low_memory=False, usecols=["id", "pull_date"])
    assert len(check) == len(df_combined), "Read-back check failed."

    print(f"✅ Saved {len(df_combined):,} rows (+{len(df_combined) - len(df_existing):,} new) | "
          f"{df_combined['pull_date'].nunique()} pull dates | {df_combined['id'].nunique()} videos")
    print(f"   File: {save_path}")
    return df_combined

<h2 style="color: #8B0000;"> Step 6: Main Execution</h2>


In [8]:
def main(run_date=None):
    """
    - Connects to YouTube API
    - Fetches ALL video IDs from Cristiano Ronaldo's channel
    - Retrieves video details (one pull_date for the whole run)
    - Appends to the existing dataset and saves
    """
    run_date = run_date or datetime.now(timezone.utc).date().isoformat()
    youtube = get_youtube_api()

    print(f"📡 Pull date (UTC): {run_date}. Fetching all videos...")
    video_ids = get_channel_videos(youtube, CHANNEL_ID)
    if not video_ids:
        raise RuntimeError("No videos found: check the API key / quota. Nothing was saved.")

    print(f"📊 Fetching details for {len(video_ids)} videos...")
    stats = get_video_details(youtube, video_ids, run_date)
    if not stats:
        raise RuntimeError("Video details came back empty. Nothing was saved.")

    save_stats(stats)
    print("🎯 YouTube stats fetched and saved successfully!")

<h2 style="color: #8B0000;">Step 7: Run the  main function of the Script</h2>


In [9]:
if __name__ == "__main__":
    main()

📡 Pull date (UTC): 2026-10-02. Fetching all videos...
📊 Fetching details for 150 videos...
📂 Loaded 57,275 existing rows (591 pull dates) from /kaggle/input/datasets/ahmad03038/cristiano-ronaldo-youtube-stats-data-daily-pull/cristiano_youtube_stats.csv
✅ Saved 57,275 rows (+0 new) | 591 pull dates | 152 videos
   File: /kaggle/working/cristiano_youtube_stats.csv
🎯 YouTube stats fetched and saved successfully!


<h2 style="color: #8B0000;"> Step 8: Quick Health Check of the Saved File</h2>


In [10]:
check = pd.read_csv(os.path.join(WORKING_DIR, FILENAME), low_memory=False)
print(f"Total rows: {len(check):,} | pull dates: {check['pull_date'].nunique()} | videos: {check['id'].nunique()}")
check.groupby("pull_date")["id"].nunique().tail(7).rename("videos per pull").to_frame()

Total rows: 57,275 | pull dates: 591 | videos: 152


,videos per pull
pull_date,
2026-09-26,100
2026-09-27,100
2026-09-28,100
2026-09-29,100
2026-09-30,100
2026-10-01,98
2026-10-02,150
